# GeoFire-X C8 NEWPOS2 — préparation désactivée

Un seul fit neuf seed42, quatre époques, seuil 0,78. Internet OFF. Aucun pixel 2021. Attacher les onze datasets p00 à p10 existants. Aucun seuil NEWPOS4 supplémentaire.


In [ ]:
RUN_GPU_PREFLIGHT = False
RUN_FIT = False
RUN_RECOVERY = False
import os,sys
if 'torch' in sys.modules: raise RuntimeError('Redémarrer le noyau avant CUDA_VISIBLE_DEVICES')
os.environ['CUDA_VISIBLE_DEVICES']='0'


In [ ]:
from pathlib import Path
import base64,hashlib,io,zipfile,json
DATA=base64.b64decode('...')
assert hashlib.sha256(DATA).hexdigest()=='6c4b54394dc94f6b3d8a2215041ea73462a518a3e52cfa1c9f99da246ea60f27'
ROOT=Path('/kaggle/working/geofirex-newpos2-code')
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(DATA)) as z:
    for n in z.namelist():
        assert (ROOT/n).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
sys.path.insert(0,str(ROOT/'src'))
from geofirex import newpos2_experiment as c
print(c.preflight(ROOT))


In [ ]:
EXPECTED={v['map_sha256'] for v in c.read(ROOT/c.old.REPORT/'partitions.json').values()}
MAPS=[p for p in Path('/kaggle/input').rglob('mount-map.json') if c.sha256(p) in EXPECTED]
assert len(MAPS)==11, 'Onze partitions p00 à p10 distinctes requises'
print(c.preflight(ROOT,MAPS))
print(json.dumps(c.read(ROOT/c.REPORT/'budget.json'),indent=2))


Autorisation GPU distincte obligatoire : copier les templates DISABLED dans /kaggle/working, renseigner authorized=True, opérateur/date UTC, quota actuel, durée disponible et deux destinations de sauvegarde. Le SHA du manifeste doit rester identique. Réserver 225 minutes et 7,5 heures-carte. Exécuter le précontrôle puis un seul fit. Ne jamais supprimer le verrou pour recommencer ; pas de resume.


In [ ]:
GPU_AUTH=Path('/kaggle/working/newpos2_gpu_authorization.json')
AUTH=Path('/kaggle/working/newpos2_fit_authorization.json')
OUT=Path('/kaggle/working/'+c.B_ID)
if RUN_GPU_PREFLIGHT:
    c.authorize(ROOT,GPU_AUTH,'gpu-preflight','cuda',OUT.parent)
    result=c.preflight(ROOT,MAPS,OUT.parent,'cuda',gpu=True)
    c.save(Path('/kaggle/working/newpos2_gpu_preflight.json'),result)
    print(result)
else: print('Précontrôle GPU désactivé')


In [ ]:
if RUN_FIT:
    assert RUN_GPU_PREFLIGHT and not RUN_RECOVERY
    c.fit(ROOT,MAPS,OUT,AUTH)
else: print('Fit NEWPOS2 désactivé')


Après interruption de l’évaluation seulement : restaurer TRAINING-SAFE avec son SHA, autorisation newpos2-evaluate distincte. Aucun entraînement supplémentaire. Un fit incomplet reste PARTIAL.


In [ ]:
RECOVERY_SOURCE=Path('/kaggle/working/restored_newpos2')
RECOVERY_OUT=Path('/kaggle/working/'+c.B_ID+'_EVALUATION_RECOVERY')
RECOVERY_AUTH=Path('/kaggle/working/newpos2_evaluation_authorization.json')
RECOVERY_ARCHIVE=Path('/kaggle/input/REPLACE_PRIVATE_MOUNT/newpos2-training-safe.bin')
if RUN_RECOVERY:
    assert RUN_GPU_PREFLIGHT and not RUN_FIT
    c.authorize(ROOT,RECOVERY_AUTH,'newpos2-evaluate','cuda',RECOVERY_OUT.parent)
    c.restore_candidate(ROOT,RECOVERY_ARCHIVE,RECOVERY_ARCHIVE.with_suffix('.sha256'),RECOVERY_SOURCE)
    c.evaluate_candidate(ROOT,MAPS,RECOVERY_SOURCE,RECOVERY_OUT,RECOVERY_AUTH)
else: print('Récupération évaluation seule désactivée')


In [ ]:
from IPython.display import display,FileLink
for p in sorted(Path('/kaggle/working').glob(c.B_ID+'*.zip')):
    print(c.old.verify_zip(p))
    display(FileLink(str(p)));display(FileLink(str(p.with_suffix('.sha256'))))
